#  Real-Time Bitcoin Price Anomaly Detection Using Amazon Kinesis
This notebook documents the full end-to-end process of building a real-time anomaly detection system for Bitcoin using AWS services including Amazon Kinesis, AWS Lambda, Apache Flink, Amazon S3, and QuickSight. It includes architecture overview, code snippets, AWS CLI commands, and build instructions.

## Project Components
- **Kinesis Data Stream** for ingesting real-time Bitcoin data
- **Apache Flink application** to compute rolling averages and mark anomalies
- **AWS Lambda** to apply ML model (Isolation Forest)
- **Kinesis Firehose** to archive data to S3
- **Amazon SNS** to send anomaly alerts
- **QuickSight Dashboard** for visualizing trends and anomalies

In [ ]:
# AWS CLI: Create Kinesis Stream
!aws kinesis create-stream --stream-name bitcoin-price-stream --shard-count 1
!aws kinesis create-stream --stream-name btc-processed-stream --shard-count 1

## Python: Send Real-time BTC Data to Kinesis

In [ ]:
import boto3, json, time
from datetime import datetime

client = boto3.client('kinesis')

def send_data():
    while True:
        record = {
            "price": 30000.0,
            "volume": 1.5e6,
            "timestamp": datetime.utcnow().isoformat()
        }
        client.put_record(
            StreamName='bitcoin-price-stream',
            Data=json.dumps(record),
            PartitionKey='partition-key')
        time.sleep(5)

## ML: Isolation Forest Model
Trained locally using Scikit-Learn and saved with `joblib`.

In [ ]:
from sklearn.ensemble import IsolationForest
import joblib
import numpy as np

X_train = np.random.rand(1000, 2)
model = IsolationForest(contamination=0.01)
model.fit(X_train)
joblib.dump(model, 'isolation_forest_model.pkl')

## Docker: Build Lambda Deployment Package
Use AWS's Lambda Python 3.9 image to ensure compatibility.

In [ ]:
# Inside EC2/Docker terminal:
!docker run -v $PWD:/var/task -it public.ecr.aws/lambda/python:3.9 bash
# Inside container:
pip install numpy==1.19.5 scikit-learn==0.24.2 joblib -t python/
cp handler.py python/
cd python && zip -r9 ../lambda.zip .

## AWS CLI: Deploy Lambda Function + Firehose + SNS

In [ ]:
# Create Firehose delivery stream
!aws firehose create-delivery-stream --delivery-stream-name btc-firehose-anomalies \
  --s3-destination-configuration file://firehose-config.json

# Create SNS topic
!aws sns create-topic --name btc-anomaly-alerts

# Update Lambda code
!aws lambda update-function-code --function-name btc-anomaly-detector \
  --zip-file fileb://lambda.zip

## Apache Flink Job (Java)
JAR was built using Maven and uploaded to S3 for use in the Managed Flink Application.

In [ ]:
# Maven project creation
!mvn archetype:generate -DgroupId=com.bitcoin.flink -DartifactId=btc-flink-job \
  -DarchetypeArtifactId=maven-archetype-quickstart -DinteractiveMode=false

# Package JAR
!cd btc-flink-job && mvn clean package

## QuickSight Dashboard
- Data stored in S3 via Firehose
- Athena table created over partitioned S3 structure
- Dashboard built with filters and color highlighting anomalies